# Standard LCB Bayesian Optimization example with susceptibility diagnostics

This notebook is based on `main_robust_suscept.ipynb`, but the optimization acquisition is changed from robust-LCB to standard LCB with `kappa = 1.0`. The post-optimization diagnostic still computes sigma-weighted susceptibility around the final LCB recommendation in the normalized design space `u \in [0, 1]^d`.

The susceptibility diagnostic uses only finite-difference gradients of the GP posterior mean `μ(u)` and reports both raw mean-squared gradient sensitivity and perturbation-variance-weighted relative contributions under the same perturbation scale used for the robust-LCB comparison notebook.


In [ ]:
from pathlib import Path
import lib_config as config

import os
import json
import numpy as np
import pandas as pd

import lib_gp
import lib_grad
import lib_grad_db
import lib_random
import lib_ga
import lib_ga_db
import lib_backbone
import lib_plot as plot

# desgin
import lib_RFdesign

%load_ext autoreload
%autoreload 2


In [ ]:
_config = config._loadConfig(Path("./_config.toml"))
app_config = config.initParams(_config, debug=True)


In [ ]:
backbone = lib_backbone.Backbone(config = app_config)
gp = lib_gp.GaussianProcess(config = app_config)

def build_optimizer(method, config):
    if method == "gp":
        return lib_gp.GaussianProcess(config=config)
    if method == "gradient":
        return lib_grad.GradientSearch(config=config)
    if method == "gradient_db":
        return lib_grad_db.GradientSearch(config=config)
    if method == "random":
        return lib_random.RandomSearch(config=config)
    if method == "ga":
        return lib_ga.RealCodedGA(config=config)
    if method == "ga_db":
        return lib_ga_db.RealCodedGA(config=config)
    raise ValueError(f"Unknown optimizer method: {method}")

base_dir = app_config.env.dir_base
backbone.initStorer()


In [ ]:
model_paths, model_paths_str = backbone._get_path_models()


In [ ]:
design = lib_RFdesign.ConvexBackshort(model_path=model_paths[0])
step_info = design.genStepBackshort(step_heights=(2.0, 1.5, 1.0), shrink=1.5)
design.plotStepBackshort3D(step_info)

# original smooth backshort example (commented coexistence)
# convex_backshort = design.genBackshort()
# design.plotConvex3D(convex_backshort)


In [ ]:
design = lib_RFdesign.ConvexFinshape(model_path=model_paths[1])
convex_finshpe = design.genFinshape()
design.plotProfile2D(convex_finshpe)


In [ ]:
RESULTS_FILE = str(backbone._get_dir_run() / Path(_config["io"]["filename_output"]))
TEMP_FILE = str(backbone._get_dir_run() / Path(_config["io"]["filename_temp"]))


In [ ]:
def getResult(input_params, param_names, temp_hfss_path, result_file_path):
    df_temp = pd.read_csv(temp_hfss_path)
    header_flag = not os.path.exists(result_file_path)
    
    try:
        s11_value = df_temp.iloc[-1, -1]
        result_row = dict(zip(param_names, input_params))
        result_row["S11"] = s11_value

        df_result = pd.DataFrame([result_row]) # jissainiha kokowo w/o csv de all data fame demo OK
        df_result.to_csv(result_file_path, mode='a', header=header_flag, index=False) # append master file

        try:
            os.remove(temp_hfss_path)
        except OSError:
            pass
        return True
        
    
    except Exception as e:
        print(f"[Error][getResult] Failed to process result: {e}")
        return False


In [ ]:
ROUND_DECIMALS = app_config.runtime.round_decimals

# --- A. HFSS ---
def target_hfss(_config_temp, sim_id, param_names, params):
    backbone.call_subroutine(_config_temp, sim_id, param_names, params, value_fmt=f"{{:.{ROUND_DECIMALS}f}}")
    getResult(params, param_names, TEMP_FILE, RESULTS_FILE)
    df_result = pd.read_csv(RESULTS_FILE)
    return df_result.iloc[-1]['S11'] # <--- should be defined as objective param

# --- B. Synthetic Test Function ---
def target_ackley(params):
    
    # --- 2. Calculation ---
    x = np.array(params)
    n = len(params)

    arg1 = -0.2 * np.sqrt((1.0/n) * np.sum(x**2))
    arg2 = (1.0/n) * np.sum(np.cos(2. * np.pi * x))
    
    y = -20. * np.exp(arg1) - np.exp(arg2) + 20. + np.e

    return y

def target_add(params):

    x = np.array(params)

    y = np.sum(params)

    return y

def target_griewank(params):
    
    x = np.array(params)
    n = len(params)

    sum_term = np.sum(x**2) / 4000.0

    indices = np.arange(1, n + 1)
    prod_term = np.prod(np.cos(x / np.sqrt(indices)))
    
    y = 1.0 + sum_term - prod_term
    
    return y

costFunction = target_add
costFunction = target_ackley
costFunction = target_griewank
costFunction = target_hfss

# =====================================
if costFunction == target_hfss:
    cfg = app_config.hfss
elif costFunction == target_ackley:
    cfg = app_config.test
elif costFunction == target_add:
    cfg = app_config.test
elif costFunction == target_griewank:
    cfg = app_config.test


def round_params(params, decimals=ROUND_DECIMALS):
    return np.round(np.asarray(params, dtype=float).flatten(), decimals)


def round_history_row(row, param_names, decimals=ROUND_DECIMALS):
    rounded = dict(row)
    for name in param_names:
        if name in rounded:
            rounded[name] = float(np.round(rounded[name], decimals))
    if 'S11' in rounded:
        rounded['S11'] = float(np.round(rounded['S11'], decimals))
    if 'Metric' in rounded and pd.notna(rounded['Metric']):
        rounded['Metric'] = float(np.round(rounded['Metric'], decimals))
    if 'gamma' in rounded and pd.notna(rounded['gamma']):
        rounded['gamma'] = float(np.round(rounded['gamma'], decimals))
    return rounded


LOWER_BOUNDS = cfg.lower_bounds
UPPER_BOUNDS = cfg.upper_bounds
PARAM_NAMES = cfg.param_names
N_REPT = cfg.n_repeats
N_INIT = cfg.n_init
N_SIML = cfg.n_simulation



In [ ]:
if cfg.n_simulation <= cfg.n_init:
    raise ValueError("n_simulation must be greater than n_init.")


In [ ]:
backbone._get_path_models()[0]


In [ ]:
_config_temp = {
"n_simulation": cfg.n_simulation,
"n_repeats": cfg.n_repeats,
#"param_names": cfg.param_names[:4],
#"param_units": cfg.param_units[:4],
"WATCH_DIR": str(backbone._get_dir_run()),
"INPUT_FILE": str(backbone._get_dir_run() / Path(_config["io"]["filename_input"])),
"MODEL_FILE": model_paths_str,
"RESULTS_FILE": str(backbone._get_dir_run() / Path(_config["io"]["filename_output"])),
"TEMP_FILE": str(backbone._get_dir_run() / Path(_config["io"]["filename_temp"])),
"DONE_FLAG_FILE": str(backbone._get_dir_run() / Path("hfss.done")),
}

done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
done_flag_path.unlink(missing_ok=True)

with open(base_dir / Path("_config_HFSS.json"), 'w') as f:
        json.dump(_config_temp , f, indent=4)

print(f"Temporarily updated '{base_dir / Path("_config_HFSS.json")}' with run-specific WATCH_DIR for HFSS.")


In [ ]:
def costFunctionWrapper(param_names, params,):

    params = round_params(params, decimals=ROUND_DECIMALS)
    sim_id = backbone._getSimulationID()
    
    y = costFunction(_config_temp, sim_id, param_names, params,)
       
    # output
    _newline = dict(zip(param_names, params))
    _newline['S11'] = float(np.round(y, ROUND_DECIMALS))
    _newline = round_history_row(_newline, param_names)

    return y, _newline



In [ ]:
# Optimizer

SEARCH_METHOD = "gp"
optimizer = build_optimizer(SEARCH_METHOD, app_config)

# Standard LCB acquisition settings for comparison with robust-LCB.
# The perturbation scale below is not used by the standard LCB acquisition;
# it is retained so the post-optimization validation and susceptibility
# diagnostics use the same perturbation conditions as the robust-LCB notebook.
STANDARD_KAPPA = 1.0
PERTURB_N_SAMPLES = 64
PERTURBATION_STD_RATIO = 0.01

_bounds_width = np.asarray(UPPER_BOUNDS, dtype=float) - np.asarray(LOWER_BOUNDS, dtype=float)
PERTURBATION_STD = PERTURBATION_STD_RATIO * _bounds_width
CUSTOM_PERTURBATION_STD = {
    "h1": 0.1,
    "h2": 0.1,
    "h3": 0.1,
    "h4": 0.1,
    "h5": 0.1,
    "s1": 0.01,
    "s2": 0.02,
    "s3": 0.02,
    "s4": 0.02,
    "s5": 0.02,
    "a": 0.1,
    "b": 0.1,
    "k": 0.1,
}
CUSTOM_PERTURBATION_STD_ARRAY = np.asarray([CUSTOM_PERTURBATION_STD[name] for name in PARAM_NAMES], dtype=float)
PERTURBATION_SIGMA = np.diag(CUSTOM_PERTURBATION_STD_ARRAY ** 2)
DESIGN_BOUNDS = np.vstack([np.asarray(LOWER_BOUNDS, dtype=float), np.asarray(UPPER_BOUNDS, dtype=float)]).T
STANDARD_ACQ_PARAMS = {
    "name": "lower_confidence_bound",
    "kappa": STANDARD_KAPPA,
}

def _build_hidden_row(row, routine_idx, method_name):
    hidden_row = dict(row)
    hidden_row["routine_idx"] = routine_idx
    hidden_row["method"] = method_name
    hidden_row["visibility"] = "hidden"
    return round_history_row(hidden_row, PARAM_NAMES)


def _save_hidden_history(debug_rows, repeat_idx):
    if not debug_rows:
        return None

    df_debug = pd.DataFrame(debug_rows)
    csv_path = backbone._get_dir_run() / f"debug_repeat_{repeat_idx}.csv"
    df_debug.to_csv(csv_path, index=False)
    print(f"Saved hidden debug rows to: {csv_path}")
    return csv_path


def _append_visible_row(row, visible_history, routine_idx, info):
    row_visible = round_history_row(row, PARAM_NAMES)
    row_visible['Metric'] = float(np.round(info.get('base_y', np.nan), ROUND_DECIMALS)) if pd.notna(info.get('base_y', np.nan)) else np.nan
    row_visible['gamma'] = float(np.round(info.get('gamma', info.get('length_scale', np.nan)), ROUND_DECIMALS)) if pd.notna(info.get('gamma', info.get('length_scale', np.nan))) else np.nan
    row_visible['routine_idx'] = routine_idx
    visible_history.append(row_visible)


def optBySearch(n_simulation, history_data, visible_history, hidden_history, initial_rows, lower_bound, upper_bound, active_indices=None, fixed_point=None):

    initial_count = len(initial_rows)
    total_budget = max(0, n_simulation - initial_count)
    if total_budget == 0:
        print("[search budget] no remaining simulation budget after initialization")
        return

    if not initial_rows:
        print("[search budget] no seed rows available for optimizer search")
        return

    if SEARCH_METHOD == "gradient_db":
        seed_row = dict(min(initial_rows, key=lambda row: row["S11"]))
        seed_label = "best_initial"
        remaining_budget = total_budget
        routine_idx = 1

        #print(
        #    f"[search routine] entering {routine_idx}/1 "
        #    f"(remaining budget incl. this: {remaining_budget}, seed: {seed_label}, n_init: {initial_count})"
        #)

        x_new, info = optimizer.search(
            history_data=history_data,
            param_names=PARAM_NAMES,
            lower_bounds=lower_bound,
            upper_bounds=upper_bound,
            objective_func=costFunctionWrapper,
            acq_params=STANDARD_ACQ_PARAMS if SEARCH_METHOD == "gp" else None,
            active_indices=active_indices,
            fixed_point=fixed_point,
            routine_index=routine_idx,
            routine_total=1,
            start_row=seed_row,
            maxiter=max(1, remaining_budget),
            maxfun=remaining_budget,
            max_evals=remaining_budget,
        )

        if x_new is not None:
            pre_row = dict(zip(PARAM_NAMES, np.asarray(x_new, dtype=float).flatten()))
            pre_row["S11"] = np.nan
            pre_row["phase"] = "x_new_pre_eval"
            hidden_history.append(_build_hidden_row(pre_row, routine_idx, "pre_eval"))

        method_name = info.get('method', SEARCH_METHOD)
        evaluated_rows = info.get('evaluated_rows', [])
        if not evaluated_rows and x_new is not None:
            _, fallback_row = costFunctionWrapper(PARAM_NAMES, x_new)
            evaluated_rows = [fallback_row]

        for row in evaluated_rows:
            rounded_row = round_history_row(row, PARAM_NAMES)
            history_data.append(rounded_row)
            hidden_history.append(_build_hidden_row(rounded_row, routine_idx, method_name))
            _append_visible_row(rounded_row, visible_history, routine_idx, info)

        if not evaluated_rows:
            print("[search routine] stopped at 1/1 because the optimizer produced no new evaluations")
            return

        print("[search routine] exited 1/1 (remaining budget: 0)")
        print(f"{routine_idx:<5} | {float(np.round(info.get('base_y', np.nan), ROUND_DECIMALS)):<10} | {method_name:<10} | {float(np.round(info.get('gamma', info.get('length_scale', np.nan)), ROUND_DECIMALS)) if pd.notna(info.get('gamma', info.get('length_scale', np.nan))) else np.nan}")
        return

    routine_idx = 0
    while routine_idx < total_budget:
        used_budget = max(0, len(visible_history) - initial_count)
        remaining_budget = total_budget - used_budget
        if remaining_budget <= 0:
            print(f"[search budget] exhausted after routine {routine_idx}")
            break

        routine_idx += 1
        if routine_idx <= len(initial_rows):
            seed_row = dict(initial_rows[routine_idx - 1])
            seed_label = f"initial[{routine_idx - 1}]"
        else:
            seed_row = dict(min(history_data, key=lambda row: row["S11"]))
            seed_label = "best_so_far"

        #print(
        #    f"[search routine] entering {routine_idx}/{total_budget} "
        #    f"(remaining budget incl. this: {remaining_budget}, seed: {seed_label})"
        #)

        x_new, info = optimizer.search(
            history_data=history_data,
            param_names=PARAM_NAMES,
            lower_bounds=lower_bound,
            upper_bounds=upper_bound,
            objective_func=costFunctionWrapper,
            acq_params=STANDARD_ACQ_PARAMS if SEARCH_METHOD == "gp" else None,
            active_indices=active_indices,
            fixed_point=fixed_point,
            routine_index=routine_idx,
            routine_total=total_budget,
            start_row=seed_row,
            maxiter=max(1, remaining_budget),
            maxfun=remaining_budget,
            max_evals=remaining_budget,
        )

        # 追加: x_new を先に記録（評価結果がなくても残す）
        if x_new is not None:
            pre_row = dict(zip(PARAM_NAMES, np.asarray(x_new, dtype=float).flatten()))
            pre_row["S11"] = np.nan
            pre_row["phase"] = "x_new_pre_eval"
            hidden_history.append(_build_hidden_row(pre_row, routine_idx, "pre_eval"))

        method_name = info.get('method', SEARCH_METHOD)
        evaluated_rows = info.get('evaluated_rows', [])
        if not evaluated_rows and x_new is not None:
            _, fallback_row = costFunctionWrapper(PARAM_NAMES, x_new)
            evaluated_rows = [fallback_row]

        for row in evaluated_rows:
            rounded_row = round_history_row(row, PARAM_NAMES)
            history_data.append(rounded_row)
            hidden_history.append(_build_hidden_row(rounded_row, routine_idx, method_name))
            _append_visible_row(rounded_row, visible_history, routine_idx, info)

        if not evaluated_rows:
            print(
                f"[search routine] stopped at {routine_idx}/{total_budget} "
                "because the optimizer produced no new evaluations"
            )
            break

        print(
            f"[search routine] exited {routine_idx}/{total_budget} "
            f"(remaining budget: {total_budget - max(0, len(visible_history) - initial_count)})"
        )
        print(f"{routine_idx:<5} | {float(np.round(info.get('base_y', np.nan), ROUND_DECIMALS)):<10} | {method_name:<10} | {float(np.round(info.get('gamma', info.get('length_scale', np.nan)), ROUND_DECIMALS)) if pd.notna(info.get('gamma', info.get('length_scale', np.nan))) else np.nan}")




## Posterior-mean susceptibility diagnostic

The functions below operate in the normalized design coordinates `u`. When the fitted GP expects physical design variables, `NormalizedInputModel` maps normalized inputs back to physical coordinates before calling the existing `predict` method. The standard LCB acquisition itself remains separate from these diagnostics.


In [ ]:
class NormalizedInputModel:
    """Adapter that exposes posterior predictions as a function of normalized u.

    The existing GaussianProcess.predict method expects the original physical
    design variables.  This adapter lets the susceptibility functions work on
    u in [0, 1]^d by mapping u -> lower + u * (upper - lower) before prediction.
    """

    def __init__(self, model, physical_bounds):
        self.model = model
        self.physical_bounds = np.asarray(physical_bounds, dtype=float)
        if self.physical_bounds.ndim != 2 or self.physical_bounds.shape[1] != 2:
            raise ValueError("physical_bounds must have shape (d, 2).")
        self.lower = self.physical_bounds[:, 0]
        self.upper = self.physical_bounds[:, 1]
        self.width = self.upper - self.lower
        if np.any(self.width < 0):
            raise ValueError("Each upper bound must be greater than or equal to the lower bound.")

    def _to_physical(self, U):
        U = np.asarray(U, dtype=float)
        return self.lower + U * self.width

    def predict(self, U, *args, **kwargs):
        return self.model.predict(self._to_physical(U), *args, **kwargs)


def _posterior_mean(model, X):
    """Return GP posterior mean as a flat NumPy array using the existing predict API."""
    prediction = model.predict(np.asarray(X, dtype=float))
    if isinstance(prediction, tuple):
        prediction = prediction[0]
    return np.asarray(prediction, dtype=float).reshape(-1)


def make_perturbed_points(u_star, epsilons, bounds):
    """Create clipped perturbation points Z = u_star + epsilons in normalized space."""
    u_star = np.asarray(u_star, dtype=float).reshape(-1)
    epsilons = np.asarray(epsilons, dtype=float)
    bounds = np.asarray(bounds, dtype=float)

    if epsilons.ndim != 2 or epsilons.shape[1] != u_star.size:
        raise ValueError("epsilons must have shape (M, d), matching u_star.")
    if bounds.shape != (u_star.size, 2):
        raise ValueError("bounds must have shape (d, 2), matching u_star.")

    Z = u_star[None, :] + epsilons
    return np.clip(Z, bounds[:, 0], bounds[:, 1])


def finite_difference_gradient_mean(model, Z, bounds, h):
    """Finite-difference gradient of GP posterior mean μ(u) at points Z."""
    Z = np.asarray(Z, dtype=float)
    bounds = np.asarray(bounds, dtype=float)
    h = float(h)

    if Z.ndim != 2:
        raise ValueError("Z must have shape (M, d).")
    if bounds.shape != (Z.shape[1], 2):
        raise ValueError("bounds must have shape (d, 2), matching Z.")
    if h <= 0:
        raise ValueError("h must be positive.")

    M, d = Z.shape
    G = np.zeros((M, d), dtype=float)

    for i in range(d):
        Z_plus = Z.copy()
        Z_minus = Z.copy()
        Z_plus[:, i] = np.clip(Z_plus[:, i] + h, bounds[i, 0], bounds[i, 1])
        Z_minus[:, i] = np.clip(Z_minus[:, i] - h, bounds[i, 0], bounds[i, 1])

        denom = Z_plus[:, i] - Z_minus[:, i]
        mu_plus = _posterior_mean(model, Z_plus)
        mu_minus = _posterior_mean(model, Z_minus)

        valid = denom != 0.0
        G[valid, i] = (mu_plus[valid] - mu_minus[valid]) / denom[valid]
        G[~valid, i] = 0.0

    return G


def compute_sigma_weighted_susceptibility(G, sigma_vec, normalize=True):
    """Compute raw and sigma-weighted susceptibility from mean gradients."""
    G = np.asarray(G, dtype=float)
    if G.ndim != 2:
        raise ValueError("G must have shape (M, d).")

    sigma_vec = np.asarray(sigma_vec, dtype=float).reshape(-1)
    if sigma_vec.shape != (G.shape[1],):
        raise ValueError("sigma_vec must have shape (d,), matching G.")
    if np.any(sigma_vec < 0):
        raise ValueError("sigma_vec values must be non-negative.")

    S_raw = np.mean(G ** 2, axis=0)
    C_sigma = (sigma_vec ** 2) * S_raw

    if normalize:
        sum_C = float(np.sum(C_sigma))
        sum_S = float(np.sum(S_raw))
        C_sigma_normalized = C_sigma / sum_C if sum_C > 0 else np.zeros_like(C_sigma)
        S_raw_normalized = S_raw / sum_S if sum_S > 0 else np.zeros_like(S_raw)
    else:
        C_sigma_normalized = np.zeros_like(C_sigma)
        S_raw_normalized = np.zeros_like(S_raw)

    return {
        "S_raw": S_raw,
        "C_sigma": C_sigma,
        "C_sigma_normalized": C_sigma_normalized,
        "S_raw_normalized": S_raw_normalized,
    }


def susceptibility_fd_sigma_weighted(
    model,
    u_star,
    sigma_vec=None,
    epsilons=None,
    bounds=None,
    h=1e-3,
    M=4096,
    random_state=None,
    normalize=True,
):
    """Sigma-weighted susceptibility using finite differences of posterior mean μ(u)."""
    u_star = np.asarray(u_star, dtype=float).reshape(-1)
    d = u_star.size

    if bounds is None:
        bounds = np.tile(np.array([[0.0, 1.0]], dtype=float), (d, 1))
    else:
        bounds = np.asarray(bounds, dtype=float)
    if bounds.shape != (d, 2):
        raise ValueError("bounds must have shape (d, 2), matching u_star.")

    if epsilons is None:
        if sigma_vec is None:
            raise ValueError("sigma_vec is required when epsilons is None.")
        sigma_vec = np.asarray(sigma_vec, dtype=float).reshape(-1)
        if sigma_vec.shape != (d,):
            raise ValueError("sigma_vec must have shape (d,), matching u_star.")
        if np.any(sigma_vec < 0):
            raise ValueError("sigma_vec values must be non-negative.")
        rng = np.random.default_rng(random_state)
        epsilons = rng.normal(loc=0.0, scale=sigma_vec, size=(int(M), d))
    else:
        epsilons = np.asarray(epsilons, dtype=float)
        if epsilons.ndim != 2 or epsilons.shape[1] != d:
            raise ValueError("epsilons must have shape (M, d), matching u_star.")
        if sigma_vec is None:
            if epsilons.shape[0] > 1:
                sigma_vec = np.std(epsilons, axis=0, ddof=1)
            else:
                sigma_vec = np.zeros(d, dtype=float)
        else:
            sigma_vec = np.asarray(sigma_vec, dtype=float).reshape(-1)
            if sigma_vec.shape != (d,):
                raise ValueError("sigma_vec must have shape (d,), matching u_star.")
            if np.any(sigma_vec < 0):
                raise ValueError("sigma_vec values must be non-negative.")

    Z = make_perturbed_points(u_star, epsilons, bounds)
    G = finite_difference_gradient_mean(model, Z, bounds, h)
    result = compute_sigma_weighted_susceptibility(G, sigma_vec, normalize=normalize)
    result.update({
        "u_star": u_star,
        "Z": Z,
        "G": G,
        "epsilons": epsilons,
        "sigma_vec": np.asarray(sigma_vec, dtype=float).reshape(-1),
        "h": float(h),
        "bounds": bounds,
    })
    return result



In [ ]:
import time
my_best_input = [
    [5.0, 1.503409, 2.285533, 2.441034, 4.919906, 0.0, 1.5, 1.5, 1.5, 1.5, 2.00, 3.627538, 6.00],
]

backbone.all_in_bounds(my_best_input, cfg.lower_bounds, cfg.upper_bounds)
active_indices, fixed_point, _ = backbone._buildSamplingIndices(dims=cfg.n_params, param_groups=cfg.param_groups, group_order=getattr(cfg, "group_order", None),)

print("Active Indices:", active_indices)
print("Fixed Point:", np.round(fixed_point, ROUND_DECIMALS))

try:
    
    for r in range(N_REPT):
        backbone.printn(f"Starting {SEARCH_METHOD} Repeat {r + 1}/{N_REPT}")
        start = time.perf_counter()

        history_data = []
        visible_history = []
        hidden_history = []
        initial_rows = []
        
        # -------------------- initial simulation --------------------------------
        backbone.printn(f"--- Generating {N_INIT} Initial Samples ---")
        
        X_initial = backbone.LHSsampler_extended(
            dims=cfg.n_params,
            nums=cfg.n_init,
            lower_bounds=cfg.lower_bounds,
            upper_bounds=cfg.upper_bounds,
            active_indices=active_indices,
            fixed_point=fixed_point,
            fixed_points = my_best_input # you can also reduce the dim of  my best_input[Ninit:]
        )
        X_initial = np.round(X_initial, ROUND_DECIMALS)
        
        print(f"{'Iter':<5} | {'New y':<10} | {'Method':<10} | {'Metric'}")        
        for i in range(N_INIT):
            params = X_initial[i]

            # add evaluation points
            pre_row = dict(zip(PARAM_NAMES, np.asarray(params, dtype=float).flatten()))
            pre_row["S11"] = np.nan
            pre_row["phase"] = "init_pre_eval"
            hidden_history.append(_build_hidden_row(pre_row, 0, "pre_eval"))

            y_new, _newline = costFunctionWrapper(PARAM_NAMES, params,)
            _newline['Metric'] = np.nan
            _newline['gamma'] = np.nan
            _newline['routine_idx'] = 0
            _newline = round_history_row(_newline, PARAM_NAMES)
            history_data.append(_newline)
            visible_history.append(_newline)
            initial_rows.append(dict(_newline))


        # -------------------- Search --------------------------------
        optBySearch(
            N_SIML,
            history_data,
            visible_history,
            hidden_history,
            initial_rows,
            LOWER_BOUNDS,
            UPPER_BOUNDS,
            active_indices=active_indices,
            fixed_point=fixed_point,
        )

        # ==============================================================================
        elapsed = time.perf_counter() - start
        print(f"Repeat {i} completed in {elapsed:.3f} seconds.")
        
        df_final = pd.DataFrame(visible_history)
        X_train = df_final[PARAM_NAMES].values
        y_train = df_final['S11'].values

        best_idx_final = np.argmin(y_train)
        print("-" * 75)
        print(f"Optimization Finished.")
        print(f"Global Best Found: y = {float(np.round(y_train[best_idx_final], ROUND_DECIMALS)):.10f}")
        
        best_x_str = np.array2string(np.round(X_train[best_idx_final], ROUND_DECIMALS), precision=ROUND_DECIMALS, separator=', ')
        print(f"At location: x = {best_x_str}")
        
        # --- Robust posterior validation summary ---
        # Fit a final GP to the nominal observations only.  No f(x + delta)
        # observations are added to the training data.
        gp_validation = lib_gp.GaussianProcess(config=app_config)
        gp_validation.fit(X_train, y_train)

        best_observed_x = X_train[best_idx_final]
        final_standard_lcb_x = np.asarray([visible_history[-1][name] for name in PARAM_NAMES], dtype=float)
        candidates = {
            "best_observed": best_observed_x,
            "standard_lcb": final_standard_lcb_x,
        }
        validation_summary = lib_gp.robust_validation_summary(
            candidates=candidates,
            gp=gp_validation,
            Sigma=PERTURBATION_SIGMA,
            bounds=DESIGN_BOUNDS,
            rng=np.random.default_rng(202),
            n_perturb=PERTURB_N_SAMPLES,
        )
        robust_validation_df = pd.DataFrame(
            {k: v for k, v in validation_summary.items() if k != "robust_best"}
        ).T

        print("\nRobust validation summary:")
        display(robust_validation_df)
        print("Robust best candidate:", validation_summary.get("robust_best"))

        # --- Local susceptibility around the standard-LCB recommendation ---
        # Diagnostics are computed in normalized coordinates u in [0, 1]^d.
        # The standard LCB optimizer above still uses the original physical bounds.
        physical_bounds = DESIGN_BOUNDS
        normalized_bounds = np.tile(np.array([[0.0, 1.0]], dtype=float), (len(PARAM_NAMES), 1))
        physical_width = physical_bounds[:, 1] - physical_bounds[:, 0]
        if np.any(physical_width <= 0):
            raise ValueError("All physical bounds must have positive width for normalized susceptibility diagnostics.")

        u_standard = (final_standard_lcb_x - physical_bounds[:, 0]) / physical_width
        sigma_vec = CUSTOM_PERTURBATION_STD_ARRAY / physical_width
        susceptibility_model = NormalizedInputModel(gp_validation, physical_bounds)

        susceptibility_result = susceptibility_fd_sigma_weighted(
            model=susceptibility_model,
            u_star=u_standard,
            sigma_vec=sigma_vec,
            epsilons=None,  # Pass externally generated epsilons here if they are available.
            bounds=normalized_bounds,
            h=1e-3,
            M=4096,
            random_state=0,
            normalize=True,
        )
        susceptibility_df = pd.DataFrame({
            "param": PARAM_NAMES,
            "sigma_u": susceptibility_result["sigma_vec"],
            "S_raw": susceptibility_result["S_raw"],
            "S_raw_normalized": susceptibility_result["S_raw_normalized"],
            "C_sigma": susceptibility_result["C_sigma"],
            "C_sigma_normalized": susceptibility_result["C_sigma_normalized"],
        })

        print("\nSigma-weighted susceptibility around standard_lcb (normalized u-space):")
        display(susceptibility_df)
        
        # --- After each repeat, archive results and save plot data ---
        df_output = backbone._genOutputDataFrame(df_final)
        df_output[PARAM_NAMES] = df_output[PARAM_NAMES].round(ROUND_DECIMALS)
        if 'S11' in df_output:
            df_output['S11'] = df_output['S11'].round(ROUND_DECIMALS)
        if 'Metric' in df_output:
            df_output['Metric'] = df_output['Metric'].round(ROUND_DECIMALS)
        if 'gamma' in df_output:
            df_output['gamma'] = df_output['gamma'].round(ROUND_DECIMALS)
        _save_hidden_history(hidden_history, r + 1)
        
        # save
        backbone._addNewDatasetToHDF(df_output, "output", f"repeat_{r+1}")
        
        # --- 5a. Visualize learning curve of the final model ---
        plot.plot_learning_curve(df_output)
        
                
finally:
    done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
    done_flag_path.touch()

    json_file = base_dir / Path("_config_HFSS.json")
    json_file.unlink(missing_ok=True) # delite the json file
    if backbone.h5f:
        backbone.h5f.close()




